Try the ott vmap tutorial

In [1]:
!hostname

cpusrv86.scidom.de


In [2]:
import time

import jax
import jax.numpy as jnp

import matplotlib.pyplot as plt

from ott.geometry import geometry, pointcloud
from ott.solvers import linear
from ott.solvers.linear import acceleration

In [3]:
import io

import requests

import numpy as np

response = requests.get("https://marcocuturi.net/embeddings.npz")
data = np.load(io.BytesIO(response.content))

In [4]:
X, HIST = data.get("X"), data.get("HIST")

In [5]:
# X contains 4000 word embeddings in dimension 50 , HIST a 653 x 4000 (row-normalized) matrix of histograms.
print(
    f"{HIST.shape[0]} texts supported on up to {HIST.shape[1]} words of dimension {X.shape[1]}"
)

653 texts supported on up to 4000 words of dimension 50


In [6]:
geom = pointcloud.PointCloud(X)
print(
    "median cost:",
    geom.median_cost_matrix,
    " mean cost:",
    geom.mean_cost_matrix,
)

median cost: 0.40351653  mean cost: 0.41272348


In [7]:
cost = geom.cost_matrix
print(" max:", jnp.max(geom.cost_matrix))

 max: 1.4388261


In [8]:
print("Default epsilon is: ", geom.epsilon)

Default epsilon is:  0.0070406534


In [9]:
epsilon = 1e-2

In [8]:
### runs for 1min48s
# n_iters = []
# for i in range(13):
#     n_iters.append(
#         linear.solve(
#             geometry.Geometry(cost_matrix=cost, epsilon=epsilon),
#             lse_mode=False,
#             a=HIST[0, :],
#             b=HIST[1 + i, :],
#         ).n_iters
#     )
# print(np.array(n_iters))

In [9]:
def sink(a, b, cost, epsilon, min_iterations, max_iterations):
    return linear.solve(
        geometry.Geometry(cost_matrix=cost, epsilon=epsilon),
        a=a,
        b=b,
        lse_mode=False,
        min_iterations=min_iterations,
        max_iterations=max_iterations,
    ).reg_ot_cost

In [10]:
sink_2vmap = jax.jit(
    jax.vmap(jax.vmap(sink, [0] + [None] * 5, 0), [None, 0] + [None] * 4, 1),
    static_argnums=[4, 5],
)

In [11]:
HIST_a = jnp.array(HIST[0:45])
HIST_b = jnp.array(HIST[-37:])
print(HIST_a.shape, HIST_b.shape, cost.shape)

(45, 4000) (37, 4000) (4000, 4000)


In [12]:
DIV = sink_2vmap(HIST_a, HIST_b, cost, 1, 0, 100)

In [ ]:
### runs > 10min, didnt wait until done
# DIS, ran_in = [], []
# epsilons = [None, 1e-2, 1e-1]
# for epsilon in epsilons:
#     tic = time.perf_counter()
#     DIS.append(
#         sink_2vmap(HIST_a, HIST_b, cost, epsilon, 0, 100).block_until_ready()
#     )
#     toc = time.perf_counter()
#     ran_in.append(toc - tic)

In [13]:
# fig, axes = plt.subplots(1, 3, figsize=(12, 6))
# fig.tight_layout()
# axes = [axes[0], axes[1], axes[2]]
# vmin = min([jnp.min(dis) for dis in DIS])
# vmax = max([jnp.max(dis) for dis in DIS])

# for epsilon, dis, ran_in_, ax_ in zip(epsilons, DIS, ran_in, axes):
#     im = ax_.imshow(dis, vmin=vmin, vmax=vmax)
#     eps = f" ({geom.epsilon:.4f})" if epsilon is None else ""
#     ax_.set_title(
#         r"$\varepsilon$ = " + str(epsilon) + eps + f"\n {ran_in_:.2f} s"
#     )
#     ax_.axis("off")

# fig.subplots_adjust(right=0.8)
# cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
# fig.colorbar(im, cax=cbar_ax)

# plt.show()

In [ ]:
epsilon = 1e-2
# Naive Vmapping
%time out_1 = sink_2vmap(HIST_a, HIST_b, cost, epsilon, 0, 100).block_until_ready()

In [ ]:
# Vmapping while forcing the number of iterations to be fixed.
%time out_2 = sink_2vmap(HIST_a, HIST_b, cost, epsilon, 100, 100).block_until_ready()

In [ ]:
jnp.linalg.norm(out_1 - out_2)